In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob, os
from google.colab import files

# remove any old copies, so we never read a stale file
for f in glob.glob("train_set*.csv") + glob.glob("test_set*.csv"):
    os.remove(f)

uploaded = files.upload()   # select BOTH train_set.csv and test_set.csv

train_name = [k for k in uploaded if k.startswith("train_set")][0]
test_name  = [k for k in uploaded if k.startswith("test_set")][0]

train = pd.read_csv(train_name, index_col=0)
test  = pd.read_csv(test_name,  index_col=0)

target = "default_next_month"
X_train, y_train = train.drop(columns=[target]), train[target]
X_test,  y_test  = test.drop(columns=[target]),  test[target]

print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 4), round(y_test.mean(), 4))

Saving train_set.csv to train_set.csv
Saving test_set.csv to test_set.csv
(24000, 28) (6000, 28)
0.2212 0.2212


In [2]:
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

print(len(num_cols), "numeric columns")
print(len(cat_cols), "categorical columns")
print(num_cols)

25 numeric columns
3 categorical columns
['LIMIT_BAL', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'months_late', 'zero_pay_months', 'avg_bill', 'avg_payment', 'utilisation']


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

In [4]:
X_train_prep = preprocessor.fit_transform(X_train)   # learns from TRAIN only
X_test_prep  = preprocessor.transform(X_test)        # only applies what was learned

print(X_train_prep.shape, X_test_prep.shape)
print()

print("LIMIT_BAL scaled, train: mean", round(X_train_prep[:, 0].mean(), 4),
      "std", round(X_train_prep[:, 0].std(), 4))
print("LIMIT_BAL scaled, test:  mean", round(X_test_prep[:, 0].mean(), 4),
      "std", round(X_test_prep[:, 0].std(), 4))
print()
print(preprocessor.get_feature_names_out()[25:])

(24000, 35) (6000, 35)

LIMIT_BAL scaled, train: mean 0.0 std 1.0
LIMIT_BAL scaled, test:  mean 0.0046 std 1.0091

['cat__SEX_1' 'cat__SEX_2' 'cat__EDUCATION_1' 'cat__EDUCATION_2'
 'cat__EDUCATION_3' 'cat__EDUCATION_4' 'cat__MARRIAGE_0' 'cat__MARRIAGE_1'
 'cat__MARRIAGE_2' 'cat__MARRIAGE_3']


In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = baseline.predict(X_train)

print("Accuracy:", round(accuracy_score(y_train, pred), 4))
print("Recall:  ", recall_score(y_train, pred))
print("ROC-AUC: ", roc_auc_score(y_train, baseline.predict_proba(X_train)[:, 1]))

Accuracy: 0.7788
Recall:   0.0
ROC-AUC:  0.5


In [6]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

print("Weight for class 0 (no default):", round(weights[0], 4))
print("Weight for class 1 (default):   ", round(weights[1], 4))
print("Ratio:", round(weights[1] / weights[0], 2))

Weight for class 0 (no default): 0.642
Weight for class 1 (default):    2.2603
Ratio: 3.52


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

log_reg = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["roc_auc", "recall", "precision", "f1", "accuracy"]

cv_lr = cross_validate(log_reg, X_train, y_train, cv=cv, scoring=scoring)

cv_results = {}   # we will fill this with every model for the comparison
cv_results["Logistic Regression"] = {m: cv_lr[f"test_{m}"].mean() for m in scoring}

for m in scoring:
    print(f"{m:10s} mean {cv_lr[f'test_{m}'].mean():.4f}   std {cv_lr[f'test_{m}'].std():.4f}")

roc_auc    mean 0.7596   std 0.0065
recall     mean 0.5997   std 0.0149
precision  mean 0.4677   std 0.0109
f1         mean 0.5255   std 0.0119
accuracy   mean 0.7605   std 0.0064


In [9]:
from sklearn.tree import DecisionTreeClassifier

free_tree = DecisionTreeClassifier(class_weight="balanced", random_state=42)

cv_free = cross_validate(free_tree, X_train, y_train, cv=cv,
                         scoring=["roc_auc", "f1"], return_train_score=True)

print("ROC-AUC  train:", round(cv_free["train_roc_auc"].mean(), 4),
      "| validation:", round(cv_free["test_roc_auc"].mean(), 4))
print("F1       train:", round(cv_free["train_f1"].mean(), 4),
      "| validation:", round(cv_free["test_f1"].mean(), 4))
print("Depth:", free_tree.fit(X_train, y_train).get_depth())

ROC-AUC  train: 1.0 | validation: 0.6143
F1       train: 0.9989 | validation: 0.3991
Depth: 44


In [10]:
tree_results = {}

for depth in [3, 4, 5, 6, 8, None]:
    tree = DecisionTreeClassifier(class_weight="balanced", max_depth=depth,
                                  min_samples_leaf=50, random_state=42)
    res = cross_validate(tree, X_train, y_train, cv=cv, scoring=scoring)
    tree_results[depth] = {m: res[f"test_{m}"].mean() for m in scoring}
    r = tree_results[depth]
    print(f"depth {str(depth):>4}  roc_auc {r['roc_auc']:.4f}  recall {r['recall']:.4f}  "
          f"precision {r['precision']:.4f}  f1 {r['f1']:.4f}")

depth    3  roc_auc 0.7624  recall 0.6643  precision 0.4253  f1 0.5185
depth    4  roc_auc 0.7669  recall 0.5854  precision 0.4899  f1 0.5324
depth    5  roc_auc 0.7688  recall 0.6092  precision 0.4649  f1 0.5268
depth    6  roc_auc 0.7672  recall 0.6310  precision 0.4513  f1 0.5259
depth    8  roc_auc 0.7610  recall 0.6480  precision 0.4230  f1 0.5117
depth None  roc_auc 0.7489  recall 0.6666  precision 0.3875  f1 0.4900


In [11]:
cv_results["Decision Tree"] = tree_results[5]

for m in scoring:
    print(f"{m:10s} {cv_results['Decision Tree'][m]:.4f}")

roc_auc    0.7688
recall     0.6092
precision  0.4649
f1         0.5268
accuracy   0.7577


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_results = {}

for depth in [6, 8, 10]:
    rf = RandomForestClassifier(n_estimators=200, max_depth=depth, min_samples_leaf=50,
                                class_weight="balanced", random_state=42, n_jobs=-1)
    res = cross_validate(rf, X_train, y_train, cv=cv, scoring=scoring)
    rf_results[depth] = {m: res[f"test_{m}"].mean() for m in scoring}
    r = rf_results[depth]
    print(f"depth {depth:>2}  roc_auc {r['roc_auc']:.4f}  recall {r['recall']:.4f}  "
          f"precision {r['precision']:.4f}  f1 {r['f1']:.4f}")

depth  6  roc_auc 0.7816  recall 0.6340  precision 0.4682  f1 0.5385


In [ ]:
cv_results["Random Forest"] = rf_results[10]

for m in scoring:
    print(f"{m:10s} {cv_results['Random Forest'][m]:.4f}")

In [ ]:
comparison = pd.DataFrame(cv_results).T[["roc_auc", "recall", "precision", "f1", "accuracy"]]
comparison = comparison.sort_values("roc_auc", ascending=False).round(4)

print(comparison)
comparison.to_csv("model_comparison_cv.csv")

In [ ]:
import matplotlib.pyplot as plt

comparison[["roc_auc", "recall", "precision", "f1"]].plot(kind="bar", figsize=(9, 5), rot=0)
plt.title("Cross-validated performance by model (training set)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             f1_score, accuracy_score, confusion_matrix)

final_rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=50,
                                  class_weight="balanced", random_state=42, n_jobs=-1)
final_rf.fit(X_train, y_train)

test_pred = final_rf.predict(X_test)
test_prob = final_rf.predict_proba(X_test)[:, 1]

print("ROC-AUC:  ", round(roc_auc_score(y_test, test_prob), 4))
print("Recall:   ", round(recall_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred), 4))
print("F1:       ", round(f1_score(y_test, test_pred), 4))
print("Accuracy: ", round(accuracy_score(y_test, test_pred), 4))
print()
print(confusion_matrix(y_test, test_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred, display_labels=["No default", "Default"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Confusion matrix (test set)")

RocCurveDisplay.from_predictions(y_test, test_prob, name="Random Forest", ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[1].set_title("ROC curve (test set)")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(final_rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

print(importances.head(10).round(3))

importances.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 feature importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# 1. Model comparison (cross-validation on the training set)
model_comparison = (comparison.reset_index()
    .rename(columns={"index": "model_name", "precision": "precision_score", "f1": "f1_score"})
    [["model_name", "roc_auc", "recall", "precision_score", "f1_score", "accuracy"]])

# 2. Final test-set results of the selected model
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
model_test_results = pd.DataFrame([{
    "model_name": "Random Forest",
    "roc_auc": round(roc_auc_score(y_test, test_prob), 4),
    "recall": round(recall_score(y_test, test_pred), 4),
    "precision_score": round(precision_score(y_test, test_pred), 4),
    "f1_score": round(f1_score(y_test, test_pred), 4),
    "accuracy": round(accuracy_score(y_test, test_pred), 4),
    "true_negatives": tn, "false_positives": fp,
    "false_negatives": fn, "true_positives": tp
}])

# 3. Feature importance (all 28 features)
feature_importance = importances.rename_axis("feature_name").reset_index(name="importance")
feature_importance["importance"] = feature_importance["importance"].round(6)
feature_importance["importance_rank"] = range(1, len(feature_importance) + 1)

# 4. Predictions for the 6,000 test customers
model_predictions = pd.DataFrame({
    "customer_id": X_test.index + 1,          # row 0 is customer ID 1
    "actual_default": y_test.values,
    "predicted_probability": test_prob.round(6),
    "predicted_default": test_pred
})
model_predictions["outcome"] = np.select(
    [(model_predictions.actual_default == 0) & (model_predictions.predicted_default == 0),
     (model_predictions.actual_default == 0) & (model_predictions.predicted_default == 1),
     (model_predictions.actual_default == 1) & (model_predictions.predicted_default == 0)],
    ["True negative", "False positive", "False negative"],
    default="True positive")

for name, table in [("model_comparison", model_comparison),
                    ("model_test_results", model_test_results),
                    ("feature_importance", feature_importance),
                    ("model_predictions", model_predictions)]:
    table.to_csv(f"{name}.csv", index=False)
    files.download(f"{name}.csv")
    print(name, table.shape)

print()
print(model_predictions["outcome"].value_counts())

joblib.dump(final_rf, "random_forest_final.joblib")
files.download("random_forest_final.joblib")